This notebook contains code for the KD baselines, KD-Imitate ZS and KD-Critique

### Finetuning GPT-2 for Imitation Phase

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel, AdamW, AutoTokenizer
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

# Set random seed
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

In [ ]:
train_data = pd.read_csv('mistral_instruct_v01_punchlines_train_20k_output.csv')

class CustomDataset(Dataset):
    def __init__(self, setups, punchlines):
        self.setups = setups
        self.punchlines = punchlines

    def __len__(self):
        return len(self.setups)

    def __getitem__(self, idx):
        setup = self.setups[idx]
        punchline = self.punchlines[idx]
        return setup, punchline

train_pairs = []

for _, row in train_data.iterrows():
    setup = row['setup']
    punchlines = [row.get(f"generated_punchline_{i}") for i in range(1, 4)]
    for punchline in punchlines:
        if pd.notna(punchline):
            train_pairs.append((setup, punchline))

# Unzip the pairs if you need separate lists again
train_setups, train_punchlines = zip(*train_pairs)

print("len(train_setups)", len(train_setups))
print("len(train_punchlines)", len(train_punchlines))

train_dataset = CustomDataset(train_setups, train_punchlines)

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_name = "gpt2"
model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

optim = torch.optim.Adam(model.parameters(), lr=2e-5) 
num_epochs = 15
train_loss_log = [] # after every log_interval iterations
epochs_train_loss = []
epochs_val_loss = []

cnt = 0
log_interval = 40

for epoch in range(num_epochs):
    
    # Training Loop
    curr_train_loss = []
    model.train()
    for batch in tqdm(train_dataloader):
        setups, punchlines = batch

        encoded_setups = [tokenizer.encode(setup, return_tensors="pt") for setup in setups]
        encoded_punchlines = [tokenizer.encode(punchline, return_tensors="pt") for punchline in punchlines]

        loss = []
        outputs = []

        for setup, punchline in zip(encoded_setups, encoded_punchlines):

            input = torch.cat((setup, punchline), dim=1).to(device)
            preds = model(input, labels=input)
            loss.append(preds.loss)

        loss = torch.stack(loss).mean()

        optim.zero_grad()
        loss.backward()
        optim.step()

        cnt += 1
        if cnt % log_interval == 0:
            print(f"Batch {cnt}, Training Loss: {loss.item()}")
            train_loss_log.append(loss.item())

        curr_train_loss.append(loss.item())

    # Logging the average training loss for the epoch
    avg_train_loss = np.mean(curr_train_loss)
    epochs_train_loss.append(avg_train_loss)
    print(f"Epoch {epoch+1}/{num_epochs} Training Loss: {avg_train_loss}")

    # Save model at the end of each epoch
    model.save_pretrained(f'models/gpt2_imitation_phase_epoch_{epoch+1}')
    
with open('logs/finetune_gpt2_train_loss_log.txt', 'w') as f:
    for item in train_loss_log:
        f.write("%s\n" % item)

with open('logs/finetune_gpt2_epochs_train_loss.txt', 'w') as f:
    for item in epochs_train_loss:
        f.write("%s\n" % item)

### Generating outputs from Finetuned GPT-2 for DPO

In [3]:
import numpy as np
import pandas as pd
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

def generate_punchlines_batched(
    model_path, 
    output_csv_path, 
    test_data_path='../data/train_set_20k.csv',
    seed=42,
    batch_size=32,
    max_gen_length=50,
    temperature=0.7,
    top_k=50,
    top_p=0.95
):
    """
    Generates punchlines using a pretrained GPT-2 model with batched processing.
    """
    # Set random seed
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

    # Set the device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Load model and tokenizer
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2', padding_side='left')
    model = GPT2LMHeadModel.from_pretrained(model_path).to(device)
    model.eval()
    
    # Add padding token
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = model.config.eos_token_id

    # Load test data
    test_data = pd.read_csv(test_data_path)
    setups = test_data['setup'].tolist()
    punchlines = test_data['punchline'].tolist()

    # Initialize results DataFrame
    result_df = pd.DataFrame(columns=['setup', 'ground_truth_punchline', 'generated_punchline'])

    # Process in batches
    for i in tqdm(range(0, len(setups), batch_size)):
        batch_setups = setups[i:i + batch_size]
        batch_punchlines = punchlines[i:i + batch_size]

        # Tokenize batch
        encoded_input = tokenizer(
            batch_setups,
            padding=True,
            return_tensors='pt'
        )

        # Generate punchlines for batch
        with torch.no_grad():
            outputs = model.generate(
                **{k: v.to(device) for k, v in encoded_input.items()},
                max_new_tokens=max_gen_length,
                temperature=temperature,
                do_sample=True,
                top_k=top_k,
                top_p=top_p,
                pad_token_id=tokenizer.pad_token_id,
                num_return_sequences=1
            )

        # Decode generated outputs
        generated_punchlines = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        # Store results for batch
        batch_results = {
            'setup': batch_setups,
            'ground_truth_punchline': batch_punchlines,
            'generated_punchline': generated_punchlines
        }
        result_df = pd.concat([
            result_df, 
            pd.DataFrame(batch_results)
        ], ignore_index=True)

        # Clear GPU cache periodically
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Save results
    result_df.to_csv(output_csv_path, index=False)
    print(f"Results saved to {output_csv_path}")

In [ ]:
# Finetuned GPT-2
for i in range(5):
    generate_punchlines_batched('./models/gpt2_imitation_phase_epoch_15', f'./outputs/gpt2_imitation_phase_output_seed_{i}.csv', seed = i)

### Get preference from teacher for DPO

In [1]:
import os
import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    GenerationConfig,
    LlamaTokenizer
)
from Levenshtein import distance as levenshtein_distance # Use python-Levenshtein
from tqdm.auto import tqdm
from itertools import combinations
import re
import logging
import gc # Garbage collector

# --- Configuration ---
STUDENT_OUTPUT_DIR = './outputs/'
STUDENT_OUTPUT_PREFIX = 'gpt2_imitation_phase_output_seed_'
NUM_SEEDS = 5
TEACHER_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.1" # 
OUTPUT_DPO_CSV_PATH = './dpo_preference_data.csv'
TEACHER_BATCH_SIZE = 8 # Adjust based on GPU memory
MAX_TEACHER_NEW_TOKENS = 50 # Enough for "Reason: ... Answer: X"
SEED = 42

# --- Logging ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [ ]:
# --- Set Seed ---
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --- Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {device}")

# --- Function to load and combine student outputs ---
def load_student_outputs(output_dir, prefix, num_seeds):
    all_dfs = []
    logger.info("Loading student outputs...")
    for i in range(num_seeds):
        file_path = os.path.join(output_dir, f"{prefix}{i}.csv")
        if os.path.exists(file_path):
            try:
                df = pd.read_csv(file_path)
                df['seed'] = i # Keep track of the source seed
                # Clean the generated punchline - remove the setup if it's prepended
                # Check if 'setup' column exists before applying lambda
                if 'setup' in df.columns and 'generated_punchline' in df.columns:
                    df['cleaned_generated'] = df.apply(
                        lambda row: row['generated_punchline'].replace(row['setup'], '', 1).strip()
                        if isinstance(row['setup'], str) and isinstance(row['generated_punchline'], str) and row['generated_punchline'].startswith(row['setup'])
                        else row['generated_punchline'],
                        axis=1
                    )
                else:
                     logger.warning(f"'setup' or 'generated_punchline' column missing in {file_path}. Using raw generated punchline.")
                     df['cleaned_generated'] = df['generated_punchline'] # Fallback

                all_dfs.append(df[['setup', 'cleaned_generated', 'seed']]) # Keep only necessary columns
                logger.info(f"Loaded {file_path}")
            except Exception as e:
                logger.error(f"Error loading or processing {file_path}: {e}")
        else:
            logger.warning(f"File not found: {file_path}")

    if not all_dfs:
        logger.error("No student output files loaded. Exiting.")
        # exit()
        assert False

    combined_df = pd.concat(all_dfs, ignore_index=True)
    logger.info(f"Combined student outputs. Total rows: {len(combined_df)}")
    # Handle potential NaN values introduced during loading/cleaning
    combined_df.dropna(subset=['setup', 'cleaned_generated'], inplace=True)
    combined_df['cleaned_generated'] = combined_df['cleaned_generated'].astype(str) # Ensure string type
    logger.info(f"Rows after dropping NaNs in setup/cleaned_generated: {len(combined_df)}")
    return combined_df

# --- Function to select diverse pairs ---
def select_diverse_pairs(group):
    candidates = group['cleaned_generated'].unique()
    if len(candidates) < 2:
        return None # Not enough unique candidates to form a pair

    max_dist = -1
    best_pair = None

    for p1, p2 in combinations(candidates, 2):
        # Ensure candidates are strings before calculating distance
        if not isinstance(p1, str) or not isinstance(p2, str):
             logger.warning(f"Skipping non-string candidate pair: ({type(p1)}, {type(p2)})")
             continue

        dist = levenshtein_distance(p1, p2)
        if dist > max_dist:
            max_dist = dist
            best_pair = (p1, p2) # Order doesn't matter here, will assign later

    return best_pair

In [ ]:
from huggingface_hub import login
login(token="<placeholder>")

# --- Teacher Model Loading ---
logger.info(f"Loading teacher model: {TEACHER_MODEL_NAME}")
try:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    teacher_model = AutoModelForCausalLM.from_pretrained(
        TEACHER_MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
    ).eval()
    teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_NAME)
    if teacher_tokenizer.pad_token is None:
        teacher_tokenizer.pad_token = teacher_tokenizer.eos_token
    logger.info("Teacher model loaded successfully.")
except Exception as e:
    logger.error(f"Error loading teacher model: {e}")
    logger.error("Ensure transformers, accelerate, bitsandbytes are installed and you have enough VRAM.")
    exit()

# --- Teacher Prompt Template ---
# Adjusted slightly for clarity, but matches user request
teacher_prompt_template = """You will be given an input text (the setup). You will then be given two paraphrase choices (potential punchlines) labeled 1 and 2 for this input text.

Choose the better paraphrase based on how humorous and human-like it sounds while being relevant to the input text.

Input text: {setup}
Paraphrases:
1. {p1}
2. {p2}

Format your answer EXACTLY like this:
Reason: [Your reason, 15-20 words maximum]
Answer: [Your choice, either 1 or 2]"""


# --- Teacher Generation Config ---
# Focused on getting the short structured answer
teacher_gen_config = GenerationConfig(
    max_new_tokens=MAX_TEACHER_NEW_TOKENS,
    temperature=0.1, # Low temp for less randomness in structured output
    top_p=1.0,
    do_sample=False, # No sampling needed for this task
    pad_token_id=teacher_tokenizer.pad_token_id,
    eos_token_id=teacher_tokenizer.eos_token_id,
    num_return_sequences=1
)

# --- Parsing Function ---
def parse_teacher_response(response_text):
    """Parses the teacher's response to extract the chosen index (1 or 2)."""
    try:
        # Use regex to find "Answer: 1" or "Answer: 2", allowing for variations
        match = re.search(r'Answer:\s*([12])', response_text, re.IGNORECASE)
        if match:
            choice = int(match.group(1))
            return choice
        else:
            # Fallback: check if the response ends with just '1' or '2'
            last_char = response_text.strip()[-1]
            if last_char in ['1', '2']:
                 logger.warning(f"Using fallback parsing (last char) for: {response_text}")
                 return int(last_char)
            else:
                logger.warning(f"Could not parse choice from response: {response_text}")
                return None
    except Exception as e:
        logger.error(f"Error parsing response '{response_text}': {e}")
        return None

In [ ]:
# --- Main Processing ---
student_data = load_student_outputs(STUDENT_OUTPUT_DIR, STUDENT_OUTPUT_PREFIX, NUM_SEEDS)

# Group by setup and find diverse pairs
logger.info("Grouping by setup and selecting diverse pairs...")
grouped = student_data.groupby('setup')
pairs_to_process = []
skipped_setups = 0

for name, group in tqdm(grouped, desc="Selecting Pairs"):
    pair = select_diverse_pairs(group)
    if pair:
        pairs_to_process.append({"setup": name, "p1": pair[0], "p2": pair[1]})
    else:
        skipped_setups += 1

logger.info(f"Selected {len(pairs_to_process)} pairs for teacher evaluation.")
logger.info(f"Skipped {skipped_setups} setups due to insufficient unique candidates.")

# Process pairs in batches to get teacher preferences
dpo_data = []
logger.info("Starting teacher preference evaluation...")

for i in tqdm(range(0, len(pairs_to_process), TEACHER_BATCH_SIZE), desc="Evaluating Batches"):
    batch = pairs_to_process[i : i + TEACHER_BATCH_SIZE]
    if not batch:
        continue

    prompts = [
        teacher_prompt_template.format(setup=item['setup'], p1=item['p1'], p2=item['p2'])
        for item in batch
    ]

    inputs = teacher_tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512 # Adjust if needed based on setup/punchline length
    ).to(device)

    try:
        with torch.no_grad():
            outputs = teacher_model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                generation_config=teacher_gen_config
            )

        # Decode *only* the generated part
        decoded_responses = []
        input_lengths = inputs["input_ids"].shape[1]
        # Ensure outputs have at least input_lengths + 1 tokens before slicing
        if outputs.shape[1] > input_lengths:
             generated_ids = outputs[:, input_lengths:]
             decoded_responses = teacher_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        else:
            # Handle case where generation is shorter than input (e.g., only EOS)
            logger.warning(f"Generation output length ({outputs.shape[1]}) not longer than input length ({input_lengths}) for batch starting at index {i}. No new tokens generated.")
            decoded_responses = [""] * len(batch) # Assign empty responses


        # Process batch results
        for idx, raw_response in enumerate(decoded_responses):
            item = batch[idx]
            parsed_choice = parse_teacher_response(raw_response)

            if parsed_choice == 1:
                chosen = item['p1']
                rejected = item['p2']
            elif parsed_choice == 2:
                chosen = item['p2']
                rejected = item['p1']
            else:
                # Could not determine preference, skip this pair
                logger.warning(f"Skipping pair for setup '{item['setup'][:50]}...' due to unparseable response: {raw_response}")
                continue

            # Ensure chosen and rejected are strings
            if not isinstance(chosen, str) or not isinstance(rejected, str):
                 logger.warning(f"Skipping pair for setup '{item['setup'][:50]}...' because chosen/rejected not strings ({type(chosen)}, {type(rejected)})")
                 continue


            dpo_data.append({
                "prompt": item['setup'], # The original input setup
                "chosen": chosen,
                "rejected": rejected
            })

    except Exception as e:
        logger.error(f"Error during teacher inference for batch starting at index {i}: {e}")
        # Potentially add more robust error handling here if needed

    # Clean up GPU memory after each batch
    del inputs, outputs
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()


# --- Save DPO Data ---
if dpo_data:
    dpo_df = pd.DataFrame(dpo_data)
    dpo_df.to_csv(OUTPUT_DPO_CSV_PATH, index=False)
    logger.info(f"Successfully saved {len(dpo_df)} preference pairs to {OUTPUT_DPO_CSV_PATH}")
else:
    logger.warning("No DPO data was generated. Check logs for errors.")

logger.info("Script finished.")

### DPOTrainer on GPT-2

In [1]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    # TrainingArguments # No longer needed
)
from trl import DPOTrainer, DPOConfig # Import DPOConfig
import logging

# --- Configuration ---
model_path = "./models/gpt2_imitation_phase_epoch_15" # Your imitation phase model
dpo_dataset_path = "./dpo_preference_data.csv"
output_dir = "./models/gpt2_dpo_final_trl" # Changed output dir slightly

In [ ]:
# --- DPO Configuration ---
# Use DPOConfig and migrate relevant arguments
# Ensure all required parameters are set here
training_args = DPOConfig(
    # --- DPO Specific ---
    beta=0.2,                   # DPO regularization parameter
    loss_type="sigmoid",        # Standard DPO loss type
    max_prompt_length=128,      # Max length for prompt portion
    max_length=256,             # Max length for prompt + completion
    # label_pad_token_id=-100,  # Default is -100, usually no need to set explicitly
    # precompute_ref_log_probs=False, # Set to True if you want to precompute (requires more memory upfront)

    # --- Standard Training Args ---
    output_dir=output_dir,
    num_train_epochs=4,         # DPO often needs fewer epochs
    per_device_train_batch_size=4, # Adjust based on VRAM
    gradient_accumulation_steps=4, # Effective batch size = 2 * 4 = 8
    gradient_checkpointing=True, # Saves VRAM, slows down training slightly
    learning_rate=4e-5,         # Lower LR often better for DPO
    lr_scheduler_type="cosine",
    warmup_steps=50,
    optim="adamw_torch",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to="tensorboard",    # Logging destination
    logging_strategy="steps",
    logging_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,         # Keep only last 2 checkpoints
    remove_unused_columns=False,# MUST be False for DPOTrainer
    push_to_hub=False,
    seed=42,                    # Set seed for reproducibility
    eval_steps=400,             # Evaluate every 400 steps
)

# --- Logging ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# --- Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {device}")
if not torch.cuda.is_available():
    logger.warning("CUDA not available, training on CPU will be very slow.")

In [ ]:
# --- Load Tokenizer ---
logger.info(f"Loading tokenizer from: {model_path}")
try:
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    if tokenizer.pad_token is None:
        logger.warning("Tokenizer does not have a pad token. Setting to eos_token.")
        tokenizer.pad_token = tokenizer.eos_token
    # DPO generally works best with left padding for decoder-only models
    tokenizer.padding_side = "left"
    logger.info("Tokenizer loaded successfully.")
except Exception as e:
    logger.error(f"Failed to load tokenizer from {model_path}: {e}")
    exit()

# --- Load and Prepare Dataset ---
logger.info(f"Loading DPO dataset from: {dpo_dataset_path}")
try:
    # Use load_dataset to load the CSV directly
    raw_dataset = load_dataset("csv", data_files=dpo_dataset_path, split="train")

    # Verify required columns exist
    required_cols = ['prompt', 'chosen', 'rejected']
    if not all(col in raw_dataset.column_names for col in required_cols):
        raise ValueError(f"Dataset must contain columns: {required_cols}. Found: {raw_dataset.column_names}")

    # DPOTrainer expects columns named 'prompt', 'chosen', 'rejected'
    # No reformatting needed if CSV columns match.
    # Filter out rows where any required field is missing or not a string (basic cleaning)
    initial_len = len(raw_dataset)
    train_dataset = raw_dataset.filter(
        lambda x: x["prompt"] is not None and isinstance(x["prompt"], str) and \
                  x["chosen"] is not None and isinstance(x["chosen"], str) and \
                  x["rejected"] is not None and isinstance(x["rejected"], str)
    )
    if len(train_dataset) < initial_len:
        logger.warning(f"Filtered out {initial_len - len(train_dataset)} rows with missing/invalid data.")

    if len(train_dataset) == 0:
         raise ValueError("Dataset is empty after filtering.")

    logger.info(f"Dataset loaded and validated. Number of preference pairs: {len(train_dataset)}")

    # Optional: Split dataset if you want evaluation
    train_test_split = train_dataset.train_test_split(test_size=0.1, seed=training_args.data_seed or 42)
    train_dataset = train_test_split["train"]
    eval_dataset = train_test_split["test"] # Pass this to DPOTrainer if needed

except FileNotFoundError:
    logger.error(f"Dataset file not found at: {dpo_dataset_path}")
    # exit()
    assert False
except ValueError as e:
    logger.error(f"Error loading or processing dataset: {e}")
    # exit()
    assert False
except Exception as e:
    logger.error(f"An unexpected error occurred during dataset loading: {e}")
    # exit()
    assert False

In [ ]:
# --- Load Model ---
# Load only the base model you want to train.
# DPOTrainer will create the reference model copy internally if ref_model=None.
logger.info(f"Loading base model for training from: {model_path}")
try:
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        low_cpu_mem_usage=True,
        torch_dtype=torch.bfloat16 if training_args.bf16 else (torch.float16 if training_args.fp16 else torch.float32),
        # quantization_config=bnb_config # Optional: Add if needed
    )
    # Ensure model's pad token ID is set if needed (DPOTrainer might handle this)
    if model.config.pad_token_id is None:
         model.config.pad_token_id = tokenizer.pad_token_id
    logger.info("Base model loaded successfully.")

except Exception as e:
     logger.error(f"Failed to load base model from {model_path}: {e}")
     exit()


# --- Initialize DPOTrainer ---
logger.info("Initializing DPOTrainer...")
try:
    dpo_trainer = DPOTrainer(
        model=model,
        ref_model=None, # Let trainer create the reference model internally
        args=training_args, # Pass the DPOConfig object here
        train_dataset=train_dataset,
        eval_dataset=eval_dataset, # Pass eval dataset if you created one
        processing_class=tokenizer,
        # beta, max_prompt_length etc. are now set within DPOConfig (training_args)
        # peft_config=peft_config, # Add LoRA/PEFT config here if desired
    )
    logger.info("DPOTrainer initialized successfully.")
except Exception as e:
    logger.error(f"Failed to initialize DPOTrainer: {e}", exc_info=True)
    exit()

In [ ]:
# --- Start Training ---
logger.info("Starting DPO training...")
try:
    training_results = dpo_trainer.train()
    logger.info("Training finished.")

    # --- Save Final Model ---
    final_save_path = os.path.join(training_args.output_dir, "final_checkpoint")
    logger.info(f"Saving final model checkpoint to {final_save_path}")
    # Use save_model method of the trainer
    dpo_trainer.save_model(final_save_path)
    # Make sure tokenizer is also saved
    tokenizer.save_pretrained(final_save_path)
    logger.info("Final model and tokenizer saved successfully.")

    # Log and save metrics
    metrics = training_results.metrics
    dpo_trainer.log_metrics("train", metrics)
    dpo_trainer.save_metrics("train", metrics)
    dpo_trainer.save_state()

except Exception as e:
    logger.error(f"An error occurred during DPO training: {e}", exc_info=True)

logger.info("Script finished.")